# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Schema URL:** https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
- **DOI:** 10.71728/senscience.vcs2-05nj

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### Record Sets in the Dataset
Below, we enumerate available record sets, their `@id`, and included fields and columns.

In [ ]:
# List available record sets with their `@id` and fields
record_sets = dataset.metadata.record_sets
if record_sets:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('fields', [])
        for f in fields:
            print(f"  Field @id: {f['@id']} - name: {f.get('name', f['@id'])}")
        columns = rs.get('columns', [])
        for c in columns:
            print(f"  Column @id: {c['@id']} - name: {c.get('name', c['@id'])}")
        print()
else:
    print("No record sets available in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** Replace the example `@id` values below with those printed in the overview above.

In [ ]:
# Example: Extract data from all available record sets
# Collect record set @ids from metadata
record_set_ids = []
if dataset.metadata.record_sets:
    for rs in dataset.metadata.record_sets:
        record_set_ids.append(rs['@id'])

# Load each record set as a DataFrame
dataframes = {}
for rs_id in record_set_ids:
    # Load records for the record set using its @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print columns of the first data frame (if available)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"DataFrame columns for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Example fields** (replace with relevant `@id`s from your dataset):
- Numeric field: `http://senscience.ai/field/score_GAD7`
- Group field: `http://senscience.ai/field/gender`

In [ ]:
# Example EDA: filter and normalize a numeric field and group by a categorical field
# Replace these @id with actual IDs from your dataset

record_set_id = record_set_ids[0] if record_set_ids else None
# Example field @id for GAD-7 score (replace as needed)
numeric_field = 'score_GAD7'  # e.g., column name from the dataframe
# Example group field @id for gender (replace as needed)
group_field = 'gender'  # e.g., column name from the dataframe

df = dataframes[record_set_id]
if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    # Group the filtered data by group_field
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print(f"Numeric field '{numeric_field}' not found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of GAD-7 scores and explore their relationship with gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of GAD-7 scores
if record_set_id and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    # Plot mean GAD-7 score by gender
    if group_field in df.columns:
        plt.figure(figsize=(6, 4))
        sns.barplot(x=group_field, y=numeric_field, data=df)
        plt.title('Mean GAD-7 Score by Gender')
        plt.xlabel('Gender')
        plt.ylabel('Mean GAD-7 Score')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides insightful measurements of mental health indicators among residents of Kilifi County, Kenya.
- Exploratory analysis of GAD-7 scores showed their distribution and potential variation across gender groups.
- This notebook demonstrates the use of the Croissant schema and the `mlcroissant` library for easy, reproducible, metadata-driven data access and analysis.

For further exploration, consider analyzing other fields such as PHQ-9 or PSQ scores, educational levels, or age groups using their respective field `@id`s.